<a href="https://colab.research.google.com/github/marcnadeau/psychic-octo-broccoli/blob/main/nb/Llama3_(8B)-Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playlist LoRA training

Ce notebook prépare le dataset Spotify local, entraîne un adaptateur LoRA avec Unsloth sur un runtime Colab GPU, puis sauvegarde l'adaptateur. Les étapes Ollama et GGUF sont volontairement séparées du training.

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-mini-4k-instruct",   # modèle léger 2.2B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "v_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.9.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


<a name="Data"></a>
### Data preparation

The JSONL files are generated locally from the Spotify Million Playlist Dataset and use the columns `instruction`, `input`, and `output`.

In [8]:
from google.colab import drive

drive.mount("/content/drive")

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/Colab Notebooks/dataset/train.jsonl",
    split="train",
)

validation_dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/Colab Notebooks/dataset/validation.jsonl",
    split="train",
)

print(dataset.column_names)
print(len(dataset))
print(len(validation_dataset))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 3.0M
-rw------- 1 root root 2.7M Sep 12 23:07 train.jsonl
-rw------- 1 root root 308K Sep 12 23:07 validation.jsonl


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset/train.jsonl",
    split="train",
)
validation_dataset = load_dataset(
    "json",
    data_files="dataset/validation.jsonl",
    split="train",
)

print(dataset.column_names)
print(len(dataset), len(validation_dataset))

In [ ]:
from unsloth import apply_chat_template, standardize_sharegpt, to_sharegpt

chat_template = """Below are instructions that describe a task.

### Instruction:
{INPUT}

### Response:
{OUTPUT}"""

def prepare_dataset(data):
    data = to_sharegpt(
        data,
        merged_prompt="{instruction}[[\nYour input is:\n{input}]]",
        output_column_name="output",
        conversation_extension=1,
    )
    data = standardize_sharegpt(data)
    return apply_chat_template(
        data,
        tokenizer=tokenizer,
        chat_template=chat_template,
    )

dataset = prepare_dataset(dataset)
validation_dataset = prepare_dataset(validation_dataset)
print(dataset.column_names)
print(dataset[0]["text"][:500])

<a name="Train"></a>
### Train the model

Le trainer démarre avec 60 steps pour vérifier rapidement le pipeline. Après ce test, remplace `max_steps=60` par `max_steps=-1` et ajoute `num_train_epochs=1` pour un entraînement complet.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=validation_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=30,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

### Export local optionnel

Le dossier `llama_lora` contient seulement l'adaptateur LoRA. Pour l'utiliser dans une application locale, exporte plutôt un fichier GGUF.

In [ ]:
# Export GGUF pour l'application locale ou Ollama
model.save_pretrained_gguf(
    "playlist_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)